In [1]:
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/risk_dataset_v3.csv")

print(df.shape)
print(df["risk"].value_counts())
df.head()

(270, 7)
risk
Medium    90
Low       90
High      90
Name: count, dtype: int64


,license,license_family,project_type,distribution_model,usage,risk,reason
0,EPL-2.0,Weak Copyleft,robotics control system,distributed,frontend component,Medium,File-level copyleft obligations create moderat...
1,MIT,Permissive,banking middleware,hosted,plugin framework,Low,Low risk because the license allows proprietar...
2,MIT,Permissive,AI inference platform,hosted,utility package,Low,License creates minimal compliance burden for ...
3,Proprietary,Restricted,robotics control system,internal-only,workflow engine,High,Strong copyleft obligations may require source...
4,GPL-3.0,Strong Copyleft,cloud CRM platform,hosted,data processing module,High,High risk because network or redistribution ob...


In [2]:
def build_user_prompt(row):
    return f"""
You are an OSS compliance assistant.

Classify this scanner output.

Package: example-package
Version: 1.0.0
Ecosystem: python
Package Manager: pip
License: {row["license"]}
License Family: {row["license_family"]}
Dependency Scope: runtime
Direct or Transitive: direct
Development Dependency: no
Project Type: {row["project_type"]}
Distribution Model: {row["distribution_model"]}
Usage: {row["usage"]}
Linking Type: dynamic
Network Exposed: yes
Commercial Use: yes
Attribution Notice: preserved
License Text: included
Source Modified: no
Redistribution: no
License Confidence: high

Return exactly:
Risk: Low/Medium/High
Reason: one short sentence
""".strip()

In [3]:
def build_assistant_response(row):
    return f"""Risk: {row["risk"]}
Reason: {row["reason"]}"""

In [4]:
records = []

for _, row in df.iterrows():
    records.append({
        "messages": [
            {
                "role": "user",
                "content": build_user_prompt(row)
            },
            {
                "role": "assistant",
                "content": build_assistant_response(row)
            }
        ]
    })

print("Total records:", len(records))
records[0]

Total records: 270


{'messages': [{'role': 'user',
   'content': 'You are an OSS compliance assistant.\n\nClassify this scanner output.\n\nPackage: example-package\nVersion: 1.0.0\nEcosystem: python\nPackage Manager: pip\nLicense: EPL-2.0\nLicense Family: Weak Copyleft\nDependency Scope: runtime\nDirect or Transitive: direct\nDevelopment Dependency: no\nProject Type: robotics control system\nDistribution Model: distributed\nUsage: frontend component\nLinking Type: dynamic\nNetwork Exposed: yes\nCommercial Use: yes\nAttribution Notice: preserved\nLicense Text: included\nSource Modified: no\nRedistribution: no\nLicense Confidence: high\n\nReturn exactly:\nRisk: Low/Medium/High\nReason: one short sentence'},
  {'role': 'assistant',
   'content': 'Risk: Medium\nReason: File-level copyleft obligations create moderate compliance requirements'}]}

In [5]:
output_path = Path("../data/qwen_scanner_finetune_dataset.jsonl")

with open(output_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", output_path)

Saved: ../data/qwen_scanner_finetune_dataset.jsonl


In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["risk"]
)

print("Train:", train_df.shape)
print(train_df["risk"].value_counts())

print("\nValidation:", val_df.shape)
print(val_df["risk"].value_counts())

Train: (216, 7)
risk
Low       72
Medium    72
High      72
Name: count, dtype: int64

Validation: (54, 7)
risk
Low       18
High      18
Medium    18
Name: count, dtype: int64


In [7]:
def save_jsonl(dataframe, path):
    records = []

    for _, row in dataframe.iterrows():
        records.append({
            "messages": [
                {
                    "role": "user",
                    "content": build_user_prompt(row)
                },
                {
                    "role": "assistant",
                    "content": build_assistant_response(row)
                }
            ]
        })

    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} records to {path}")


save_jsonl(train_df, "../data/qwen_scanner_finetune_train.jsonl")
save_jsonl(val_df, "../data/qwen_scanner_finetune_val.jsonl")

Saved 216 records to ../data/qwen_scanner_finetune_train.jsonl
Saved 54 records to ../data/qwen_scanner_finetune_val.jsonl
